In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "3"
import textattack
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd
from langdetect import detect

/home/kokil/shaz/interp-toxicity/interp-toxicity/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-25 14:10:10.607847: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-25 14:10:10.626687: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753452610.644590 1926644 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753452610.649849 1926644 

In [2]:
from tqdm import tqdm, trange

from datasets import load_dataset
import pandas as pd
import functools
import sys
from pathlib import Path
from typing import Callable

# import circuitsvis as cv
import einops
import numpy as np
import torch as t
import torch.nn as nn
import torch.nn.functional as F
import eindex
# from IPython.display import display
from jaxtyping import Float, Int
from torch import Tensor
from tqdm import tqdm
# from transformer_lens import (
#     ActivationCache,
#     FactoredMatrix,
#     HookedTransformer,
#     HookedTransformerConfig,
#     HookedEncoderDecoder,
#     HookedEncoder,
#     utils,
# )
# from transformer_lens.hook_points import HookPoint

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoTokenizer
# from transformer_lens import HookedTransformer
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
tqdm.pandas()

In [3]:
print("Available CUDA devices:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}: {torch.cuda.get_device_name(i)}")
device = torch.device('cuda:5') if torch.cuda.is_available() else torch.device('cpu')

Available CUDA devices: 8
Device 0: NVIDIA H100 80GB HBM3
Device 1: NVIDIA H100 80GB HBM3
Device 2: NVIDIA H100 80GB HBM3
Device 3: NVIDIA H100 80GB HBM3
Device 4: NVIDIA H100 80GB HBM3
Device 5: NVIDIA H100 80GB HBM3
Device 6: NVIDIA H100 80GB HBM3
Device 7: NVIDIA H100 80GB HBM3


In [4]:
tokenizer = AutoTokenizer.from_pretrained("s-nlp/roberta_toxicity_classifier")
model = AutoModelForSequenceClassification.from_pretrained("s-nlp/roberta_toxicity_classifier")

Some weights of the model checkpoint at s-nlp/roberta_toxicity_classifier were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


## Baseline

In [15]:
df = pd.read_csv('jigsaw/test_clean.csv')

In [9]:
from sklearn.metrics import accuracy_score
from tqdm import trange

# Batch size for GPU inference
BATCH_SIZE = 64

all_preds = []
all_labels = []

model = model.to(device)
model.eval()
new_df = df.sample(5000)
num_samples = len(new_df)
for start_idx in trange(0, num_samples, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, num_samples)
    batch_texts = new_df.comment_text.iloc[start_idx:end_idx].tolist()
    batch_labels = new_df.toxic.iloc[start_idx:end_idx].astype(int).tolist()
    inputs = tokenizer(batch_texts, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1).cpu().tolist()
    all_preds.extend(preds)
    all_labels.extend(batch_labels)

accuracy = accuracy_score(all_labels, all_preds)
print(f"Baseline model accuracy on test set: {accuracy:.4f}")

  0%|          | 0/79 [00:00<?, ?it/s]


AttributeError: 'DataFrame' object has no attribute 'comment_text'

## Attack

In [11]:
# Import our PGD attack implementation
from pgd_bert_attack import PGDBERTAttack, set_seed

# Set seed for reproducibility
set_seed(42)

In [13]:
# Initialize PGD Attack
print("Setting up PGD BERT Attack...")

# Create the attacker - it will automatically load BERT MLM model for candidate generation
attacker = PGDBERTAttack(
    model=model,
    tokenizer=tokenizer,
    device=device,
    max_iters=10,
    top_k_tokens=5,
    mlm_top_k=50,
    sim_threshold=0.9,  # Slightly lower threshold for more flexibility
    max_length=512
)

print("PGD Attack setup complete!")


Setting up PGD BERT Attack...
Loading BERT MLM model for candidate generation...


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


PGD Attack setup complete!


In [16]:
# Evaluate model robustness on a sample of data
print("Evaluating model robustness with PGD attacks...")

# Sample some data for evaluation
sample_size = 100  # Reduced for demonstration
sample_df = df.sample(sample_size, random_state=42)
sample_texts = sample_df.comment_text.tolist()
sample_labels = sample_df.toxic.astype(int).tolist()

print(f"Running attacks on {sample_size} samples...")
print(f"Sample distribution: {sum(sample_labels)} toxic, {len(sample_labels) - sum(sample_labels)} non-toxic")

# Run batch attack evaluation with saving enabled
output_file = "snlp_roberta/jigsaw_pgd.jsonl"
robustness_results = attacker.evaluate_robustness(
    sample_texts, 
    sample_labels,
    output_file=output_file,
    log_interval=100  # Save every 25 examples
)

print(f"\nRobustness Evaluation Results:")
print(f"Total samples: {robustness_results['total_samples']}")
print(f"Successful attacks: {robustness_results['successful_attacks']}")
print(f"Attack success rate: {robustness_results['attack_success_rate']:.2%}")
print(f"Average iterations for successful attacks: {robustness_results['average_iterations']:.1f}")
print(f"Results saved to: {output_file}")


Evaluating model robustness with PGD attacks...
Running attacks on 100 samples...
Sample distribution: 10 toxic, 90 non-toxic
Evaluating robustness on 100 samples...


Attacking:   0%|          | 0/100 [00:00<?, ?it/s]

Loading sentence transformer for similarity checking...


Attacking: 100%|██████████| 100/100 [10:33<00:00,  6.34s/it]

Processed 100/100, Success rate: 37.00%
Saved intermediate results (100 examples) → snlp_roberta/jigsaw_pgd.jsonl
Saved final results (100 examples) → snlp_roberta/jigsaw_pgd.jsonl
Final attack success rate: 37.00%

Robustness Evaluation Results:
Total samples: 100
Successful attacks: 37
Attack success rate: 37.00%
Average iterations for successful attacks: 3.5
Results saved to: snlp_roberta/jigsaw_pgd.jsonl


In [17]:
# Analyze attack results and show examples
print("Analysis of Attack Results:")
print("=" * 50)

results = robustness_results['results']
successful_attacks = [(i, adv_text, meta) for i, (adv_text, meta) in enumerate(results) if meta['success']]

if successful_attacks:
    print(f"\nShowing first 5 successful attacks:")
    print("-" * 40)
    
    for i, (idx, adv_text, meta) in enumerate(successful_attacks[:5]):
        original_text = meta['original_text']
        print(f"\nExample {i+1}:")
        print(f"Original: {original_text}")
        print(f"Adversarial: {adv_text}")
        print(f"Original label: {sample_labels[idx]}")
        print(f"Iterations: {meta['iters']}")
        
        # Check the difference
        from difflib import SequenceMatcher
        similarity = SequenceMatcher(None, original_text, adv_text).ratio()
        print(f"Text similarity: {similarity:.3f}")
        print("-" * 40)
else:
    print("No successful attacks found in this sample.")

# Attack success by label
print(f"\nAttack Success by Original Label:")
toxic_attacks = [(adv_text, meta) for i, (adv_text, meta) in enumerate(results) if sample_labels[i] == 1]
non_toxic_attacks = [(adv_text, meta) for i, (adv_text, meta) in enumerate(results) if sample_labels[i] == 0]

if toxic_attacks:
    toxic_success_rate = sum(1 for _, meta in toxic_attacks if meta['success']) / len(toxic_attacks)
    print(f"Toxic examples: {toxic_success_rate:.2%} success rate ({len(toxic_attacks)} samples)")

if non_toxic_attacks:
    non_toxic_success_rate = sum(1 for _, meta in non_toxic_attacks if meta['success']) / len(non_toxic_attacks)
    print(f"Non-toxic examples: {non_toxic_success_rate:.2%} success rate ({len(non_toxic_attacks)} samples)")

print(f"\nOverall model robustness: {100 - robustness_results['attack_success_rate']*100:.1f}% robust to PGD attacks")


Analysis of Attack Results:

Showing first 5 successful attacks:
----------------------------------------

Example 1:
Original: Barnstar for you  

  stylebackgroundcolor fdffe7 border 1px solid fceb92 
 rowspan2 styleverticalalign middle padding 5px   
 stylefontsize xlarge padding 3px 3px 0 3px height 15em  The Persian Merit Barnstar 
  
 styleverticalalign middle padding 3px  For your contributions to Persian language topics
Adversarial: Barnstar for you  

  stylebackgroundcolor fdffe7 border 1px solid fceb92 
 rowspan2 styleverticalalign middle padding 5px   
 stylefontsize xlarge padding 3px 3px 0 3px height 15em  The Persian Merit Barnstar 
  
 styleverticalalign middle padding 3px  For youuck,ag!
Original label: 0
Iterations: 7
Text similarity: 0.914
----------------------------------------

Example 2:
Original: lack of Mark Lavin  New York Post contrversy in the artical  

 Its strange there is nothing in the artical about the Mark Lavin exposing oberman about his alegedly bei

In [ ]:
# Demonstrate loading and analyzing saved results
print("Loading and analyzing saved attack results...")

# You can load results from a previous run
if output_file:
    # Analyze the saved results
    analysis = attacker.analyze_saved_results(output_file)
    
    print(f"\nDetailed breakdown:")
    print(f"- Toxic examples attacked: {analysis.get('toxic_samples', 0)}")
    print(f"- Non-toxic examples attacked: {analysis.get('non_toxic_samples', 0)}")
    print(f"- Robustness score: {100 - analysis.get('attack_success_rate', 0)*100:.1f}%")
    
    # You can also manually load the raw data for custom analysis
    raw_results = attacker.load_saved_results(output_file)
    print(f"\nFirst saved result example:")
    if raw_results:
        first_result = raw_results[0]
        print(f"ID: {first_result.get('id')}")
        print(f"Original: {first_result.get('orig_text', '')[:100]}...")
        print(f"Adversarial: {first_result.get('adv_text', '')[:100]}...")
        print(f"Success: {first_result.get('success')}")
        print(f"Iterations: {first_result.get('iters')}")
else:
    print("No output file specified - skipping analysis demo")
